<a href="https://colab.research.google.com/github/isi1993/DRRR/blob/main/funnel_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The business is experiencing low conversion rates across its e-commerce funnel and wants to identify where users drop off and how to improve overall revenue performance.

In [6]:
!pip install pandasql

In [7]:
pysqldf = lambda q: sqldf(q, globals())

In [8]:
import pandas as pd
import sqlite3
from pandasql import sqldf
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from statsmodels.formula.api import ols
import statsmodels.api as sm
import numpy as np
from numpy import heaviside
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [9]:
df = pd.read_excel('/content/user_events.xlsx')
df.head()

,event_id,user_id,event_type,event_date,product_id,amount,traffic_source
0,8490,5024,page_view,2025-12-30 04:58:24.517,205,NaN,social
1,2296,1713,page_view,2025-12-30 05:10:26.276,201,NaN,organic
2,3896,2558,page_view,2025-12-30 05:11:09.002,404,NaN,organic
3,2297,1713,add_to_cart,2025-12-30 05:13:26.276,201,NaN,organic
4,2298,1713,checkout_start,2025-12-30 05:16:26.276,201,NaN,organic


In [10]:
df.isnull().sum()

,0
event_id,0
user_id,0
event_type,0
event_date,0
product_id,0
amount,8555
traffic_source,0


In [11]:

df['amount'] = df['amount'].fillna(0)

In other to have all row intact i replaced the null value in the amount column with 0 as the amount column is a numeric

In [12]:
df.shape

(9381, 7)

In [13]:
# CHEACK FOR THE VARIOUS PRODUCT FOUND IN THE DATAFRAME
df['product_id'].value_counts()

,count
product_id,
404,1646
205,1611
201,1594
305,1549
101,1516
102,1465


In [14]:
query = """
SELECT *
FROM df
"""
sqldf(query)

,event_id,user_id,event_type,event_date,product_id,amount,traffic_source
0,8490,5024,page_view,2025-12-30 04:58:24.517000,205,0.0,social
1,2296,1713,page_view,2025-12-30 05:10:26.276000,201,0.0,organic
2,3896,2558,page_view,2025-12-30 05:11:09.002000,404,0.0,organic
3,2297,1713,add_to_cart,2025-12-30 05:13:26.276000,201,0.0,organic
4,2298,1713,checkout_start,2025-12-30 05:16:26.276000,201,0.0,organic
...,...,...,...,...,...,...,...
9376,7296,4407,page_view,2026-02-03 03:59:46.554000,404,0.0,paid_ads
9377,7666,4595,checkout_start,2026-02-03 04:00:18.556000,404,0.0,organic
9378,1606,1344,add_to_cart,2026-02-03 04:03:43.775000,102,0.0,paid_ads
9379,7667,4595,payment_info,2026-02-03 04:07:18.556000,404,0.0,organic


In [15]:
df['event_date'].unique()

<DatetimeArray>
['2025-12-30 04:58:24.517000', '2025-12-30 05:10:26.276000',
 '2025-12-30 05:11:09.002000', '2025-12-30 05:13:26.276000',
 '2025-12-30 05:16:26.276000', '2025-12-30 05:20:55.417000',
 '2025-12-30 05:23:26.276000', '2025-12-30 05:25:26.276000',
 '2025-12-30 05:25:27.600000', '2025-12-30 05:29:09.002000',
 ...
 '2026-02-03 03:25:01.912000', '2026-02-03 03:36:47.170000',
 '2026-02-03 03:46:43.775000', '2026-02-03 03:56:18.556000',
 '2026-02-03 03:59:18.556000', '2026-02-03 03:59:46.554000',
 '2026-02-03 04:00:18.556000', '2026-02-03 04:03:43.775000',
 '2026-02-03 04:07:18.556000', '2026-02-03 04:10:18.556000']
Length: 9381, dtype: datetime64[ns]

In [16]:
df['event_type'].unique()

array(['page_view', 'add_to_cart', 'checkout_start', 'payment_info',
       'purchase'], dtype=object)

In [17]:
df.dtypes

,0
event_id,int64
user_id,int64
event_type,object
event_date,datetime64[ns]
product_id,int64
amount,float64
traffic_source,object


In [18]:
query = """
WITH funnel as (
  SELECT
    COUNT(DISTINCT CASE WHEN  event_type = 'page_view' THEN user_id  END) AS stage_1_views,
    COUNT(DISTINCT CASE WHEN  event_type = 'add_to_cart' THEN user_id  END) AS stage_2_cart,
    COUNT(DISTINCT CASE WHEN  event_type = 'checkout_start' THEN user_id  END) AS stage_3_checkout,
    COUNT(DISTINCT CASE WHEN  event_type = 'payment_info' THEN user_id  END) AS stage_4_payment,
    COUNT(DISTINCT CASE WHEN  event_type = 'purchase' THEN user_id  END) AS stage_5_purchase
  FROM df
  WHERE event_date >= DATETIME('now', '-120 days')
)
SELECT  *
FROM funnel
"""
pysqldf(query)

,stage_1_views,stage_2_cart,stage_3_checkout,stage_4_payment,stage_5_purchase
0,5000,1553,1103,899,826


In [19]:
# CALCULATING THE CONVERTION RATE AT VARIOUS STAGES OF THE FUNNEL
query = """
WITH funnel as (
  SELECT
    COUNT(DISTINCT CASE WHEN  event_type = 'page_view' THEN user_id  END) AS stage_1_views,
    COUNT(DISTINCT CASE WHEN  event_type = 'add_to_cart' THEN user_id  END) AS stage_2_cart,
    COUNT(DISTINCT CASE WHEN  event_type = 'checkout_start' THEN user_id  END) AS stage_3_checkout,
    COUNT(DISTINCT CASE WHEN  event_type = 'payment_info' THEN user_id  END) AS stage_4_payment,
    COUNT(DISTINCT CASE WHEN  event_type = 'purchase' THEN user_id  END) AS stage_5_purchase
  FROM df
  WHERE event_date >= DATETIME('now', '-120 days')
)
SELECT

stage_1_views,
stage_2_cart,
ROUND(stage_2_cart * 100.0  / stage_1_views) AS view_to_cart_rate,

stage_3_checkout,
ROUND(stage_3_checkout * 100   / stage_2_cart) AS cart_to_checkout_rate,

stage_4_payment,
ROUND( stage_4_payment * 100  / stage_3_checkout) AS checkout_to_payment_rate,

stage_5_purchase,
ROUND(stage_5_purchase * 100   / stage_4_payment) AS payment_to_purchase_rate,
ROUND(stage_5_purchase * 100   / stage_1_views) AS overall_conversion_rate
FROM funnel
"""
pysqldf(query)

,stage_1_views,stage_2_cart,view_to_cart_rate,stage_3_checkout,cart_to_checkout_rate,stage_4_payment,checkout_to_payment_rate,stage_5_purchase,payment_to_purchase_rate,overall_conversion_rate
0,5000,1553,31.0,1103,71.0,899,81.0,826,91.0,16.0


The largest drop-off occurs at the view-to-cart stage, where 69% of users leave
the funnel (5000 → 1553 users).
This indicates that while traffic acquisition is effective, a significant portion of users are not engaging with the product enough to add it to their cart.

In [20]:
# CALCULATE THE PERCENTAGE OF DROP AT VARIOUS STAGES OF THE FUNNEL
query = """
WITH funnel AS (
  SELECT
    COUNT(DISTINCT CASE WHEN event_type = 'page_view' THEN user_id END) AS views,
    COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart' THEN user_id END) AS cart,
    COUNT(DISTINCT CASE WHEN event_type = 'checkout_start' THEN user_id END) AS checkout,
    COUNT(DISTINCT CASE WHEN event_type = 'payment_info' THEN user_id END) AS payment,
    COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS purchase
  FROM df
),

dropoffs AS (
  SELECT 'view_to_cart' AS stage, views AS start_users, cart AS end_users,
         ROUND(100.0 * (views - cart) / views, 2) AS drop_pct
  FROM funnel

  UNION ALL

  SELECT 'cart_to_checkout', cart, checkout,
         ROUND(100.0 * (cart - checkout) / cart, 2)
  FROM funnel

  UNION ALL

  SELECT 'checkout_to_payment', checkout, payment,
         ROUND(100.0 * (checkout - payment) / checkout, 2)
  FROM funnel

  UNION ALL

  SELECT 'payment_to_purchase', payment, purchase,
         ROUND(100.0 * (payment - purchase) / payment, 2)
  FROM funnel
)

SELECT *
FROM dropoffs
ORDER BY drop_pct DESC
---LIMIT 1;"""
pysqldf(query)

,stage,start_users,end_users,drop_pct
0,view_to_cart,5000,1553,68.94
1,cart_to_checkout,1553,1103,28.98
2,checkout_to_payment,1103,899,18.50
3,payment_to_purchase,899,826,8.12


We lose the most customers at the very first step—about 69% leave without adding to cart.
After that, the drop-offs are much smaller at each stages.
This means the main issue is getting users interested enough to take action early.
Once users start the process, most of them actually complete the purchase.

In [21]:
df['event_type'].value_counts()

,count
event_type,
page_view,5000
add_to_cart,1553
checkout_start,1103
payment_info,899
purchase,826


In [22]:
df.shape

(9381, 7)

In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
# CALCULATE DROP_OFF BY TRAFFIC SOURCE
query = """
WITH SourceData AS (
  SELECT
    traffic_source,
    COUNT(DISTINCT CASE WHEN event_type = 'page_view' THEN user_id END) AS views,
    COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart' THEN user_id END) AS cart
  FROM df
  GROUP BY traffic_source
)
SELECT
  traffic_source,
  views,
  cart,
  ROUND(100.0 * (views - cart) / views, 2) AS drop_pct
FROM SourceData
ORDER BY drop_pct DESC;"""
pysqldf(query)

,traffic_source,views,cart,drop_pct
0,social,1472,200,86.41
1,organic,2038,669,67.17
2,paid_ads,968,358,63.02
3,email,522,326,37.55


Social media traffic performs the worst—86% leave without adding to cart.
Email performs best, with the lowest drop-off at 38%, meaning higher intent users.
Paid ads and organic sit in the middle but still lose over 60% early on.
Overall, the issue is largely driven by low-quality traffic, especially from social channels

In [25]:
# CALCULATE THE PERCENTAGE OF CONVERSION RATE  BETWEEN NEW AND RETURNING CUSTOMER
query = """
WITH user_summary AS (
    SELECT
        user_id,
        MIN(event_date) AS first_event_time,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS converted
    FROM df
    GROUP BY user_id
),

user_type AS (
    SELECT
        e.user_id,
        u.converted,
        CASE
            WHEN e.event_date = u.first_event_time THEN 'new'
            ELSE 'returning'
        END AS user_type
    FROM df e
    JOIN user_summary u
        ON e.user_id = u.user_id
)

SELECT
    user_type,
    AVG(converted * 1.0) AS conversion_rate
FROM user_type
GROUP BY user_type; """
pysqldf(query)

,user_type,conversion_rate
0,new,0.165200
1,returning,0.754166


From the solution above we  could tell the returning customer have  higher conversion rate at 75% while new visitor is at 16.5% posibility of make purchase

In [26]:
# CALCULATE THE NUMBER OF PRODUCT SOLD BY THEIR UNIQUE PRODUCT ID
query ="""
SELECT
    product_id,
    COUNT(*) AS total_purchases
FROM df
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY total_purchases DESC
 """
pysqldf(query)


,product_id,total_purchases
0,205,147
1,404,141
2,305,138
3,201,134
4,101,134
5,102,132


Poduct_id 205, 404 and 138 recorded the highest number of sales

In [27]:
# CALCULATE THE HOUR OF THE DAY WITH THE HIGHEST CONVERSION RATE
query = """
WITH hourly_views AS (
    SELECT
        CAST(strftime('%H', event_date) AS INT) AS hour,
        COUNT(*) AS views
    FROM df
    WHERE event_type = 'page_view'
    GROUP BY hour
),

hourly_purchases AS (
    SELECT
        CAST(strftime('%H', event_date) AS INT) AS hour,
        COUNT(*) AS purchases
    FROM df
    WHERE event_type = 'purchase'
    GROUP BY hour
)

SELECT
    v.hour,
    v.views,
    COALESCE(p.purchases, 0) AS purchases,
    COALESCE(p.purchases, 0) * 1.0 / v.views AS conversion_rate
FROM hourly_views v
LEFT JOIN hourly_purchases p
    ON v.hour = p.hour
ORDER BY v.hour; """
pysqldf(query)

,hour,views,purchases,conversion_rate
0,0,183,30,0.163934
1,1,196,24,0.122449
2,2,213,34,0.159624
3,3,233,30,0.128755
4,4,233,46,0.197425
5,5,223,38,0.170404
6,6,193,27,0.139896
7,7,213,38,0.178404
8,8,231,38,0.164502
9,9,207,34,0.164251


WE Could tell that the highest conversion occur around 2pm and 9pm we should keep track of visitors around these time period later 11pm and also by 7pm

In [28]:
# CALCULATE THE CONVERSION RATE BY WEEKDAYS
df['weekday'] = df['event_date'].dt.day_name()

query = """
WITH weekday_views AS (
    SELECT
        weekday AS weekday,
        COUNT(*) AS views
    FROM df
    WHERE event_type = 'page_view'
    GROUP BY weekday
),

weekday_purchases AS (
    SELECT
        weekday AS weekday,
        COUNT(*) AS purchases
    FROM df
    WHERE event_type = 'purchase'
    GROUP BY weekday
)

SELECT
    v.weekday,
    v.views,
    COALESCE(p.purchases, 0) AS purchases,
    COALESCE(p.purchases, 0) * 1.0 / v.views AS conversion_rate
FROM weekday_views v
LEFT JOIN weekday_purchases p
ON v.weekday = p.weekday
ORDER BY
    CASE v.weekday
        WHEN 'Monday' THEN 1
        WHEN 'Tuesday' THEN 2
        WHEN 'Wednesday' THEN 3
        WHEN 'Thursday' THEN 4
        WHEN 'Friday' THEN 5
        WHEN 'Saturday' THEN 6
        WHEN 'Sunday' THEN 7
    END;"""
pysqldf(query)

,weekday,views,purchases,conversion_rate
0,Monday,698,127,0.181948
1,Tuesday,699,108,0.154506
2,Wednesday,705,113,0.160284
3,Thursday,676,107,0.158284
4,Friday,758,123,0.162269
5,Saturday,716,116,0.162011
6,Sunday,748,132,0.176471


The page is Recording Alot of high conversion on Mondays with few percent drop other days of the week

In [29]:
# CALCULATE THE NUMBER OF PURCHASE BY TRAFIC SOURCE
query = """

  SELECT
  traffic_source,
    COUNT(DISTINCT CASE WHEN  event_type = 'page_view' THEN user_id  END) AS views,
    COUNT(DISTINCT CASE WHEN  event_type = 'add_to_cart' THEN user_id  END) AS cart,
    COUNT(DISTINCT CASE WHEN  event_type = 'purchase' THEN user_id  END) AS purchase
  FROM df
  WHERE event_date >= DATETIME('now', '-120 days')
  GROUP BY traffic_source
"""
pysqldf(query)

,traffic_source,views,cart,purchase
0,email,522,326,177
1,organic,2038,669,343
2,paid_ads,968,358,204
3,social,1472,200,102


WE COULD BE HAPPY AS MEMBER OF THE MANAGEMENT AS PEOPLE ARE BUYING OUR PRODUCT ORGANICALLY WITH 343 PRODUCT WHILE PAID_ADS AND EMAIL WHERE NOT DOING BAD AS A SOURCE OF TRAFIC SOCIAL HAS MASSIVE VIEWS BUT MANY COULD NOT ADD TO CART

In [30]:

# We Are going to calculate cart_conversion_rate,	purchase_conversion_rate,	cart_to_conversion_rate
query = """
WITH source_funnel AS (
  SELECT
    traffic_source,
    COUNT(DISTINCT CASE WHEN event_type = 'page_view' THEN user_id END) AS views,
    COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart' THEN user_id END) AS cart,
    COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS purchase
  FROM df
  WHERE event_date >= DATETIME('now', '-120 days')
  GROUP BY traffic_source
)
SELECT
  traffic_source,
  views,
  cart,
  purchase,
  ROUND(cart * 100.0 / views) AS cart_conversion_rate,
  ROUND(purchase * 100.0 / views) AS purchase_conversion_rate,
  ROUND(purchase * 100.0 / cart) AS cart_to_conversion_rate
FROM source_funnel
ORDER BY purchase DESC;
"""
pysqldf(query)

,traffic_source,views,cart,purchase,cart_conversion_rate,purchase_conversion_rate,cart_to_conversion_rate
0,organic,2038,669,343,33.0,17.0,51.0
1,paid_ads,968,358,204,37.0,21.0,57.0
2,email,522,326,177,62.0,34.0,54.0
3,social,1472,200,102,14.0,7.0,51.0


Email and paid ads bring the most valuable customers who are more likely to buy. Organic traffic brings the most people, but they are less likely to purchase immediately. Social media brings interest, but very few people convert into buyers.

In [31]:
# CALCULATE THE TIME USER SPEND ON OUR PAGE OR WEBSITE
query = """
WITH user_journey AS (
  SELECT
    user_id,
    MIN( CASE WHEN  event_type = 'page_view' THEN event_date END) AS view_time,
    MIN(CASE WHEN  event_type = 'add_to_cart' THEN event_date  END) AS cart_time,
    MIN( CASE WHEN  event_type = 'purchase' THEN event_date  END) AS purchase_time
  FROM df
  WHERE event_date >= DATETIME('now', '-82 days')
  GROUP BY user_id
  HAVING MIN( CASE WHEN  event_type = 'purchase' THEN event_date  END) IS NOT NULL
  )

SELECT
COUNT(*) as convertd_users,
ROUND(AVG((julianday(cart_time) - julianday(view_time)) * 24 * 60), 2) AS avg_view_to_cart_minutes,
ROUND(AVG((julianday(purchase_time) - julianday(cart_time)) * 24 * 60), 2) AS avg_cart_to_purchase_minutes,
ROUND(AVG((julianday(purchase_time) - julianday(view_time)) * 24 * 60), 2) AS avg_total_journey_minutes
FROM user_journey
"""
pysqldf(query)

,convertd_users,avg_view_to_cart_minutes,avg_cart_to_purchase_minutes,avg_total_journey_minutes
0,0,None,None,None


In [32]:
# revenue funnel analysis
query = """
WITH funnel_revenue AS (
  SELECT
    COUNT(DISTINCT CASE WHEN event_type = 'page_view' THEN user_id END) AS total_visitor,
    COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS total_buyers,
    SUM(CASE WHEN event_type = 'purchase' THEN amount END) AS total_revenue,
    COUNT(CASE WHEN event_type = 'purchase' THEN event_id END) AS total_order
  FROM df
  WHERE event_date >= DATETIME('now', '-120 days')
)
SELECT
  total_visitor,
  total_buyers,
  total_revenue,
  total_order,
  CASE WHEN total_order > 0 THEN total_revenue * 1.0 / total_order ELSE 0.0 END AS avg_order_value,
  CASE WHEN total_buyers > 0 THEN total_revenue * 1.0 / total_buyers ELSE 0.0 END AS revenue_per_buyer,
  CASE WHEN total_visitor > 0 THEN total_revenue * 1.0 / total_visitor ELSE 0.0 END AS revenue_per_visitor
FROM funnel_revenue
"""
pysqldf(query)

,total_visitor,total_buyers,total_revenue,total_order,avg_order_value,revenue_per_buyer,revenue_per_visitor
0,5000,826,87975.11,826,106.507397,106.507397,17.595022


In [33]:
import pandas as pd
import plotly.graph_objects as go

# Funnel data (replace with your df logic if needed)
stages = ["Visitors", "Product Page Views", "Add to Cart", "Initiate Checkout", "Purchase"]
values = [5000, 1553, 1103, 899, 826]

# Create funnel chart
fig = go.Figure(go.Funnel(
    y=stages,
    x=values,

    textinfo="value+percent initial",

    marker={
        "color": [
            "#1abc9c",  # teal
            "#2e86de",  # blue
            "#f39c12",  # orange
            "#e67e22",  # darker orange
            "#e74c3c"   # red
        ]
    }
))

# Layout styling
fig.update_layout(
    title="Conversion Funnel",
    funnelmode="stack",
)

fig.show()

we could clearly see that out 5000 who view our product only 17% endup making payment

In [34]:
df.columns

Index(['event_id', 'user_id', 'event_type', 'event_date', 'product_id',
       'amount', 'traffic_source', 'weekday'],
      dtype='object')

In [35]:
df.rename(columns={'Hours' : 'hour', 'Year': 'year', 'Week Days': 'WeekDays', 'Hours': 'hour'}, inplace=True)

In [36]:
df.columns

Index(['event_id', 'user_id', 'event_type', 'event_date', 'product_id',
       'amount', 'traffic_source', 'weekday'],
      dtype='object')

In [37]:

funnel_df = pd.DataFrame({
    "stage": ["View", "Cart", "Checkout", "Payment", "Purchase"],
    "users": [5000, 1553, 1103, 899, 826]
})

In [54]:
df.columns

Index(['event_id', 'user_id', 'event_type', 'event_date', 'product_id',
       'amount', 'traffic_source', 'weekday', 'Year', 'Months', 'Week Days',
       'Hours', 'Minute'],
      dtype='object')

In [38]:
funnel_df

,stage,users
0,View,5000
1,Cart,1553
2,Checkout,1103
3,Payment,899
4,Purchase,826


In [39]:

traffic_df = pysqldf(query)

In [40]:
values = funnel_df['users'].tolist()

In [41]:
df.columns

Index(['event_id', 'user_id', 'event_type', 'event_date', 'product_id',
       'amount', 'traffic_source', 'weekday'],
      dtype='object')

In [42]:
df['event_type'].unique()

array(['page_view', 'add_to_cart', 'checkout_start', 'payment_info',
       'purchase'], dtype=object)

In [43]:
df['event_type'] = df['event_type'].replace({'page_view': 'View', 'add_to_cart': 'Cart', 'checkout_start': 'Checkout',
'payment_info': 'Payment'})

In [57]:
df.event_type.value_counts()

,count
event_type,
View,5000
Cart,1553
Checkout,1103
Payment,899
purchase,826


In [44]:
df['event_type'].unique()

array(['View', 'Cart', 'Checkout', 'Payment', 'purchase'], dtype=object)

In [45]:
df['Year'] = df['event_date'].dt.year

In [46]:

df['Months'] = df['event_date'].dt.month_name()

In [47]:
df['Week Days'] = df['event_date'].dt.day_name()

In [48]:
df['Hours'] = df['event_date'].dt.hour

In [49]:
df['Minute'] = df['event_date'].dt.minute

In [50]:
df['Months'].value_counts()

,count
Months,
January,8287
February,624
December,470


In [51]:
df['Year'].value_counts()

,count
Year,
2026,8911
2025,470


In [52]:
df['Week Days'].value_counts()

,count
Week Days,
Sunday,1437
Friday,1422
Monday,1379
Saturday,1318
Wednesday,1292
Tuesday,1282
Thursday,1251


In [53]:
result.to_csv('funnel.csv', index=False)
from google.colab import files
files.download('funnel.csv')


NameError: name 'result' is not defined

In [ ]:
df.to_csv('expensive3_user_event.csv', index=False)
from google.colab import files
files.download('expensive3_user_event.csv')

In [ ]:
df['event_type'].unique()


Final version you can literally use in your presentation

If you want something ready-made:

“Our analysis shows that the biggest bottleneck in the funnel occurs at the View to Add-to-Cart stage, where nearly 70% of users drop off.

This drop-off is significantly higher than any other stage, while later stages show relatively strong conversion rates.

Segment analysis reveals that social traffic drives high volume but low conversion, and new users convert at just 16% compared to 75% for returning users.

This indicates that the core issue is not checkout friction, but weak early-stage engagement and lack of user trust.

Improving this stage—even slightly—could lead to a substantial increase in overall revenue.”